# **neuraTape:** Neural Turing Machines

![Neural Turing Machines Architecture](./image.png)

In [1]:
import torch

## Memory Matrix

In [2]:
N = 16          # number of memory locations
M = 4           # vector size of each memory location

In [3]:
M_t = torch.rand(N, M)  # memory matrix at time t
print(f"Memory matrix M_t: {M_t}")
print(f"Memory matrix M_t shape: {M_t.shape}")

Memory matrix M_t: tensor([[0.0074, 0.2109, 0.5753, 0.3442],
        [0.2708, 0.9342, 0.2250, 0.5770],
        [0.2705, 0.0870, 0.9586, 0.0788],
        [0.8242, 0.3791, 0.3684, 0.8519],
        [0.6990, 0.3688, 0.7200, 0.3676],
        [0.2907, 0.3001, 0.9912, 0.5856],
        [0.1647, 0.2457, 0.9896, 0.8309],
        [0.6113, 0.2594, 0.4899, 0.5541],
        [0.8754, 0.7893, 0.7550, 0.3764],
        [0.1621, 0.2942, 0.8590, 0.5088],
        [0.0838, 0.1325, 0.2717, 0.1657],
        [0.3676, 0.1646, 0.1436, 0.3749],
        [0.9719, 0.7153, 0.6100, 0.8798],
        [0.9834, 0.0744, 0.3288, 0.8896],
        [0.1829, 0.3031, 0.1915, 0.8027],
        [0.3094, 0.5680, 0.8910, 0.6866]])
Memory matrix M_t shape: torch.Size([16, 4])


## Read Head

In [4]:
r = torch.rand(N)                               # read vector at time t (N, )
w_r_t = torch.nn.functional.softmax(r, dim=0)   # read weights at time t (N, )
w_r_t = w_r_t.unsqueeze(1)                      # reshape to (N, 1)

r_t = M_t.T @ w_r_t                             # read vector at time t (M, )
print(f"Read vector r_t: {r_t}")
print(f"Read vector r_t shape: {r_t.shape}")

Read vector r_t: tensor([[0.4801],
        [0.3736],
        [0.5634],
        [0.5446]])
Read vector r_t shape: torch.Size([4, 1])


## Write Head

In [5]:
w = torch.rand(N)                               # write weights at time t (N, )
w_w_t = torch.nn.functional.softmax(w, dim=0)   # write weights at time t (N, )
w_w_t = w_w_t.unsqueeze(1)                      # reshape to (N, 1)

e_t = torch.rand(M)                             # erase vector at time t (M, )
e_t = torch.nn.functional.softmax(e_t, dim=0)   # normalize erase vector
e_t = e_t.unsqueeze(0)                          # reshape to (1, M)

a_t = torch.rand(M)                             # add vector at time t (M, )
a_t = torch.nn.functional.softmax(a_t, dim=0)   # normalize add vector
a_t = a_t.unsqueeze(0)                          # reshape to (1, M)

M_t_plus_1_e = M_t * (1 - w_w_t @ e_t)          # (N, M) * (N, M) -> (N, M)
M_t_plus_1_a = w_w_t @ a_t                      # (N, 1) @ (1, M) -> (N, M)
M_t_plus_1 = M_t_plus_1_e + M_t_plus_1_a        # (N, M) + (N, M) -> (N, M)

print(f"Updated memory matrix M_t+1: {M_t_plus_1}")
print(f"Updated memory matrix M_t+1 shape: {M_t_plus_1.shape}")

Updated memory matrix M_t+1: tensor([[0.0202, 0.2185, 0.5785, 0.3479],
        [0.2824, 0.9363, 0.2330, 0.5771],
        [0.2900, 0.1018, 0.9556, 0.0919],
        [0.8402, 0.3917, 0.3802, 0.8441],
        [0.7165, 0.3817, 0.7227, 0.3742],
        [0.3130, 0.3139, 0.9868, 0.5856],
        [0.1769, 0.2532, 0.9874, 0.8272],
        [0.6231, 0.2687, 0.4954, 0.5548],
        [0.8835, 0.7927, 0.7560, 0.3797],
        [0.1932, 0.3123, 0.8578, 0.5119],
        [0.1128, 0.1517, 0.2885, 0.1804],
        [0.3786, 0.1727, 0.1527, 0.3782],
        [0.9821, 0.7206, 0.6140, 0.8736],
        [0.9919, 0.0850, 0.3367, 0.8842],
        [0.1960, 0.3108, 0.2007, 0.7991],
        [0.3224, 0.5737, 0.8900, 0.6849]])
Updated memory matrix M_t+1 shape: torch.Size([16, 4])


## Addressing Mechanisms

### Focusing by Content

In [6]:
k_t = torch.rand(M)                             # key vector at time t (M, )
k_t = torch.unsqueeze(k_t, 0)                   # reshape to (1, M)
b_t = torch.rand(1)                             # key strength at time t (1, ) [scalar]

similarity = torch.nn.functional.cosine_similarity(M_t, k_t, dim=1)     # (N, M) vs (1, M) -> (N, )
w_c_t = torch.nn.functional.softmax(similarity * b_t, dim=0)            # (N, )
w_c_t = w_c_t.unsqueeze(1)                                              # reshape to (N, 1)

print(f"Updated content-based addressing weights w_c_t: {w_c_t}")
print(f"Updated content-based addressing weights w_c_t shape: {w_c_t.shape}")

Updated content-based addressing weights w_c_t: tensor([[0.0589],
        [0.0636],
        [0.0554],
        [0.0646],
        [0.0648],
        [0.0614],
        [0.0598],
        [0.0651],
        [0.0660],
        [0.0607],
        [0.0630],
        [0.0644],
        [0.0666],
        [0.0610],
        [0.0606],
        [0.0641]])
Updated content-based addressing weights w_c_t shape: torch.Size([16, 1])


### Focusing by Location

In [7]:
w_t_minus_1 = torch.rand(N)                     # last write weights at time t-1 (N, )
w_t_minus_1 = w_t_minus_1.unsqueeze(1)          # reshape to (N, 1)

g_t = torch.rand(1)                             # interpolation gate at time t (1, ) [scalar]
s_t = torch.rand(N)                             # shift weighting (logits) at time t (N, )
s_t = torch.softmax(s_t, dim=0)                 # shift weighting at time t (N, )
y_t = torch.rand(1)                             # sharpening factor at time t (1, ) [scalar]

w_g_t = g_t * w_c_t + (1 - g_t) * w_t_minus_1                           # (N, 1)

w_s_t = torch.zeros_like(w_g_t)                                         # (N, 1)
for i in range(N):
    w_s_t[i] = torch.sum(torch.roll(w_g_t, shifts=i) * s_t[i])          # (1, )

w_y_t = w_s_t ** y_t                                                    # (N, 1)
w_t = w_y_t / torch.sum(w_y_t)                                          # (N, 1)

print(f"Updated addressing weights w_t: {w_t}")
print(f"Updated addressing weights w_t shape: {w_t.shape}")

Updated addressing weights w_t: tensor([[0.0739],
        [0.0500],
        [0.0529],
        [0.0738],
        [0.0582],
        [0.0663],
        [0.0746],
        [0.0582],
        [0.0539],
        [0.0607],
        [0.0702],
        [0.0745],
        [0.0588],
        [0.0549],
        [0.0504],
        [0.0686]])
Updated addressing weights w_t shape: torch.Size([16, 1])


## Controller

In [8]:
N_R_HEADS = 4
N_W_HEADS = 4

INPUT_SIZE = 4
BATCH_SIZE = 4

### Feed-forward Controller

In [9]:
class FeedForwardController(torch.nn.Module):
    def __init__(self, input_size, n_r_heads, n_w_heads, head_size):
        super().__init__()

        emb_size = (n_r_heads + n_w_heads) * head_size

        self.relu = torch.nn.ReLU()
        self.ff_1 = torch.nn.Linear(input_size, input_size)
        self.norm_1 = torch.nn.LayerNorm(input_size)
        self.ff_2 = torch.nn.Linear(input_size, emb_size)
        self.norm_2 = torch.nn.LayerNorm(emb_size)
        self.ff_3 = torch.nn.Linear(emb_size, emb_size)
        self.norm_3 = torch.nn.LayerNorm(emb_size)


    def forward(self, x):
        # x: (B, input_size)

        x = self.relu(self.ff_1(x))
        x = self.norm_1(x)          # (B, input_size)

        x = self.relu(self.ff_2(x))
        x = self.norm_2(x)          # (B, emb_size)

        x = self.relu(self.ff_3(x))
        x = self.norm_3(x)          # (B, emb_size)

        return x
    
ff_controller = FeedForwardController(input_size=INPUT_SIZE, n_r_heads=N_R_HEADS, n_w_heads=N_W_HEADS, head_size=M)
print(f"FeedForwardController parameters: {sum(p.numel() for p in ff_controller.parameters()):,}")

FeedForwardController parameters: 1,372


In [10]:
x = torch.randn(BATCH_SIZE, INPUT_SIZE)
output = ff_controller(x)

print(f"FeedForwardController output: {output}")
print(f"FeedForwardController output shape: {output.shape}")

FeedForwardController output: tensor([[ 0.6912, -0.8503,  1.3857,  0.0375, -0.8503,  0.6108, -0.2321,  0.9404,
         -0.8503, -0.8503, -0.6925,  0.5907,  2.6561, -0.8503,  0.6601, -0.2010,
          1.8610, -0.8503, -0.8503, -0.8503,  0.6927, -0.8503, -0.8503,  0.8687,
         -0.8503, -0.7311, -0.8503,  1.2617, -0.2099,  1.7137, -0.8503, -0.8503],
        [-0.3043,  1.3681, -0.4033,  1.2813, -0.5828, -0.6797,  1.7153, -0.6797,
         -0.6797, -0.6797,  2.4065, -0.6577, -0.6797, -0.4633, -0.6797, -0.6797,
          2.2407, -0.6797,  1.4083,  0.9770, -0.6797, -0.6393, -0.6797,  0.6370,
         -0.6797,  0.4207,  1.4717, -0.6797, -0.6797, -0.6797, -0.6797, -0.6797],
        [-0.7436,  2.1919, -0.7436,  1.1309, -0.7436, -0.7436,  2.2750, -0.7436,
          1.0824, -0.7436,  1.5298, -0.7436, -0.7436, -0.3957, -0.7436, -0.7436,
          0.9307, -0.7436,  0.1427,  2.1514, -0.7436,  0.1598, -0.7436,  0.7124,
         -0.7436,  0.3803,  0.9492, -0.7436, -0.7436, -0.7436,  0.1438, -0.74